# Phase 1: Modeling Table Construction
## Smart Building AI — South Wing Hourly Dataset

This notebook demonstrates the reproducible construction and validation of the two primary machine learning modeling tables:
1. **Occupancy Modeling Table** (`data/processed/occupancy_data.parquet`)
2. **Energy Modeling Table** (`data/processed/energy_data.parquet`)

Both tables are restricted strictly to the verified empirical sensor overlap period (**May 22, 2018 to February 21, 2019**), resampled to **hourly resolution**, cleaned of sensor error artifacts, and feature-engineered with zero lookahead bias.

In [ ]:
import os
import sys
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Add repository root to path
module_path = os.path.abspath(os.path.join('..'))
if module_path not in sys.path:
    sys.path.append(module_path)

from src.preprocessing import build_modeling_tables

RAW_DIR = os.path.join('..', 'data', 'raw')
PROCESSED_DIR = os.path.join('..', 'data', 'processed')

print(f"Raw data directory: {RAW_DIR}")
print(f"Processed data directory: {PROCESSED_DIR}")

### 1. Build and Load Processed Modeling Tables

In [ ]:
# Execute pipeline or load cached Parquet files
occ_path = os.path.join(PROCESSED_DIR, 'occupancy_data.parquet')
energy_path = os.path.join(PROCESSED_DIR, 'energy_data.parquet')

if os.path.exists(occ_path) and os.path.exists(energy_path):
    occ_df = pd.read_parquet(occ_path)
    energy_df = pd.read_parquet(energy_path)
    print("Loaded existing processed Parquet files successfully.")
else:
    occ_df, energy_df, report = build_modeling_tables(RAW_DIR, PROCESSED_DIR)
    print("Built modeling tables from raw sources.")

print(f"Occupancy Table shape: {occ_df.shape}")
print(f"Energy Table shape: {energy_df.shape}")

### 2. Verify Table Schemas and Integrity

In [ ]:
# Integrity Assertions
for name, df in [('Occupancy Table', occ_df), ('Energy Table', energy_df)]:
    assert df['timestamp'].is_monotonic_increasing, f'{name} is not chronologically sorted!'
    assert not df['timestamp'].duplicated().any(), f'{name} contains duplicate timestamps!'
    assert df.isnull().sum().sum() == 0, f'{name} contains unexpected null values!'
    print(f"[PASS] {name}: Chronological monotonicity, uniqueness, and zero missingness verified.")

print("\n--- Occupancy Targets & Features ---")
print("Targets:", [c for c in occ_df.columns if c in ['is_occupied', 'occ_total_mean']])
print("Feature count:", len(occ_df.columns) - 3) # timestamp + 2 targets

print("\n--- Energy Targets & Features ---")
print("Targets:", [c for c in energy_df.columns if 'kwh' in c and 'lag' not in c and 'rolling' not in c])
print("Feature count:", len(energy_df.columns) - 5) # timestamp + 4 targets

### 3. Occupancy Target Analysis

In [ ]:
plt.figure(figsize=(12, 4))
plt.subplot(1, 2, 1)
sns.countplot(x='is_occupied', data=occ_df, palette='Blues_d')
plt.title('Occupancy Binary Class Distribution (0=Empty, 1=Occupied)')
plt.xlabel('is_occupied')
plt.ylabel('Hour Count')

plt.subplot(1, 2, 2)
sns.histplot(occ_df['occ_total_mean'], bins=30, kde=True, color='teal')
plt.title('Occupant Headcount Distribution')
plt.xlabel('Mean Occupant Count')
plt.ylabel('Frequency')
plt.tight_layout()
plt.show()

class_counts = occ_df['is_occupied'].value_counts(normalize=True)
print("Occupancy Class Ratios:")
print(class_counts.to_string())

### 4. Energy Consumption Breakdown

In [ ]:
energy_df['timestamp'] = pd.to_datetime(energy_df['timestamp'])
sub_sample = energy_df.set_index('timestamp').loc['2018-06-01':'2018-06-14']

plt.figure(figsize=(14, 5))
plt.plot(sub_sample.index, sub_sample['south_wing_total_kwh'], label='Total South Wing Energy (kWh)', color='black', linewidth=1.5)
plt.plot(sub_sample.index, sub_sample['hvac_S_kwh'], label='HVAC (kWh)', color='firebrick', alpha=0.8)
plt.plot(sub_sample.index, sub_sample['mels_S_kwh'], label='Plug Loads (kWh)', color='royalblue', alpha=0.8)
plt.plot(sub_sample.index, sub_sample['lig_S_kwh'], label='Lighting (kWh)', color='gold', alpha=0.9)
plt.title('Two-Week High-Resolution Energy Profile (South Wing, June 2018)')
plt.xlabel('Date')
plt.ylabel('Energy Consumption (kWh)')
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

### 5. Leakage Verification

In [ ]:
# Verify zero concurrent energy target components exist in occupancy features
occ_cols = set(occ_df.columns)
energy_targets = {'south_wing_total_kwh', 'lig_S_kwh', 'mels_S_kwh', 'hvac_S_kwh'}
assert len(occ_cols.intersection(energy_targets)) == 0, 'LEAKAGE DETECTED: Concurrent energy variables found in occupancy table!'
print('[PASS] Zero concurrent energy variables in Occupancy Table.')

# Verify energy predictors do not include unlagged target submeters
energy_predictors = [c for c in energy_df.columns if c not in energy_targets and c != 'timestamp']
assert len(set(energy_predictors).intersection(energy_targets)) == 0, 'LEAKAGE DETECTED: Concurrent target submeters found in energy features!'
print('[PASS] Zero concurrent target components in Energy Table predictors.')